# Indicator Analysis Template
**Week:** [FILL IN] | **Indicator:** [FILL IN] | **Period:** [FILL IN]

Copy this notebook for each new indicator. Fill in the sections marked [FILL IN].
Do not modify the structure — consistency across weeks is the point.

---

## Section 1 — Indicator Overview
Fill in before running any code.

**Description:** [What does this indicator do in plain English?]

**Formula:** [Key formula from Ehlers article]

**Expected behavior:** [What should the output look like on a chart?]

**NNFX role candidate:** [Baseline / C1 / C2 / Exit / ATR — your hypothesis before testing]

**Source article:** [TASC month/year or book chapter]

## Section 2 — Data Loading

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sys
import os

# Add analysis/scripts to path so we can import indicator_metrics
sys.path.append(os.path.abspath('../../analysis/scripts'))
from indicator_metrics import calculate_lag, calculate_snr, compare_fx_crypto, generate_validation_report

print('Libraries loaded successfully')

Libraries loaded successfully


In [ ]:
def load_zorro_csv(filepath: str) -> pd.DataFrame:
    """
    Loads a Zorro CSV export into a clean DataFrame.

    Handles:
    - No header row (Zorro doesn't write one reliably)
    - Date format YYYY.MM.DD
    - Warmup rows where indicator values are 0 or near-0
      (these are the first N bars before the indicator initializes)

    The warmup filter threshold of 0.001 works for FX prices (~1.10).
    For crypto exports, adjust to a higher value (e.g. 100.0).
    """
    # Read CSV — no header, so we assign column names manually
    # This matches the format: Date,Close,ITrend,SmoothPrice,EMA20
    df = pd.read_csv(
        filepath,
        header=None,
        names=['Date', 'Close', 'ITrend', 'SmoothPrice', 'EMA20'],
        parse_dates=['Date'],
        date_format='%Y.%m.%d'
    )

    df = df[df['Close'] > 0.001].copy()

    # Set Date as the index for clean time-series plotting
    df.set_index('Date', inplace=True)

    # Drop warmup period by date rather than by threshold
    # Warmup length varies by indicator — using a fixed start date is more reliable
    # All analysis uses 2015-2024 as per the roadmap
    df = df[df.index >= '2015-01-01'].copy()

    # Drop any remaining NaN rows
    df.dropna(inplace=True)

    print(f'Loaded {len(df)} bars from {df.index[0].date()} to {df.index[-1].date()}')
    return df


# --- [FILL IN] Update these paths for each new indicator ---
FX_CSV     = 'C:/Users/eyalp/Documents/fx-trading-systems/data/fx/EUR_USD_ITrend_export.csv'
CRYPTO_CSV = 'C:/Users/eyalp/Documents/fx-trading-systems/data/crypto/BTC_USD_ITrend_export.csv'
PERIOD     = 20   # [FILL IN] period used by the indicator in Zorro

# Load FX data
df_fx = load_zorro_csv(FX_CSV)
df_fx.head()

TypeError: Invalid comparison between dtype=int64 and str

In [4]:
print(df_fx[['Close', 'ITrend']].head(20).to_string())

              Close   ITrend
Date                        
2014-08-15  1.33977  0.52908
2014-08-18  1.33567  0.62070
2014-08-19  1.33162  0.65271
2014-08-20  1.32897  0.67855
2014-08-21  1.32840  0.73940
2014-08-22  1.32384  0.84529
2014-08-25  1.32009  0.96839
2014-08-26  1.31961  1.06775
2014-08-27  1.31968  1.12537
2014-08-28  1.31749  1.14851
2014-08-29  1.31506  1.16135
2014-09-01  1.31333  1.19011
2014-09-02  1.31274  1.24977
2014-09-03  1.31393  1.33850
2014-09-04  1.29608  1.43949
2014-09-05  1.29599  1.53110
2014-09-08  1.29400  1.59583
2014-09-09  1.29117  1.62586
2014-09-10  1.29027  1.62485
2014-09-11  1.29393  1.60491


## Section 3 — Visual Inspection (FX)
Look before you measure. Does the indicator behave as expected?

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(16, 8), sharex=True)

# Top panel: Price + indicator overlay
axes[0].plot(df_fx.index, df_fx['Close'],    color='black', linewidth=0.8, label='Close')
axes[0].plot(df_fx.index, df_fx['ITrend'],   color='blue',  linewidth=1.2, label='ITrend')
axes[0].plot(df_fx.index, df_fx['EMA20'],    color='green', linewidth=0.8, label='EMA20', linestyle='--')
axes[0].set_title('[FILL IN Indicator Name] vs Price — EUR/USD D1')
axes[0].legend()
axes[0].set_ylabel('Price')

# Bottom panel: SmoothPrice (or secondary output if applicable)
axes[1].plot(df_fx.index, df_fx['SmoothPrice'], color='red', linewidth=1.0, label='SmoothPrice')
axes[1].plot(df_fx.index, df_fx['Close'],        color='black', linewidth=0.5, alpha=0.4, label='Close')
axes[1].set_title('SmoothPrice component')
axes[1].legend()
axes[1].set_ylabel('Price')

plt.tight_layout()
plt.show()

# Visual checklist — answer these before moving to Section 4
print('\nVisual checklist:')
print('  [ ] Indicator follows price direction without obvious bugs')
print('  [ ] No spikes, flatlines, or discontinuities')
print('  [ ] Smoother than raw price (not identical to it)')
print('  [ ] Warmup period looks clean (no zero values in chart)')

## Section 4 — Lag Measurement (FX)
How many bars behind price is the indicator, vs a plain SMA of the same period?

In [ ]:
# Calculate lag for indicator and SMA benchmark
lag_itrend = calculate_lag(df_fx['Close'], df_fx['ITrend'])
lag_ema    = calculate_lag(df_fx['Close'], df_fx['EMA20'])

print(f'ITrend lag : {lag_itrend["lag_bars"]} bars  (correlation: {lag_itrend["correlation"]})')
print(f'EMA20  lag : {lag_ema["lag_bars"]} bars  (correlation: {lag_ema["correlation"]})')
print()

if lag_itrend['lag_bars'] <= lag_ema['lag_bars']:
    print('✅ LAG TEST PASSED — ITrend is not slower than EMA20')
else:
    diff = lag_itrend['lag_bars'] - lag_ema['lag_bars']
    print(f'❌ LAG TEST FAILED — ITrend is {diff} bars slower than EMA20')

# Plot cross-correlation curve so you can see the shape, not just the number
# A sharp peak = clean lag measurement. A flat curve = ambiguous.
correlations = []
for k in range(0, 51):
    p   = df_fx['Close'] - df_fx['Close'].mean()
    ind = df_fx['ITrend'] - df_fx['ITrend'].mean()
    if k == 0:
        correlations.append(np.corrcoef(p, ind)[0, 1])
    else:
        correlations.append(np.corrcoef(p[k:], ind[:-k])[0, 1])

plt.figure(figsize=(10, 4))
plt.plot(range(51), correlations, color='blue', linewidth=1.5)
plt.axvline(lag_itrend['lag_bars'], color='red', linestyle='--', label=f'Peak lag = {lag_itrend["lag_bars"]} bars')
plt.title('Cross-correlation curve — ITrend vs Price (EUR/USD D1)')
plt.xlabel('Lag (bars)')
plt.ylabel('Correlation')
plt.legend()
plt.tight_layout()
plt.show()

## Section 5 — Noise Reduction Score (FX)
Does the indicator reduce noise better than a plain SMA?

In [ ]:
snr_result = calculate_snr(df_fx['Close'], df_fx['ITrend'], PERIOD)

print(f'SNR — ITrend : {snr_result["snr_indicator"]}')
print(f'SNR — SMA{PERIOD}  : {snr_result["snr_sma"]}')
print(f'Improvement  : {snr_result["snr_improvement"]}x')
print()

if snr_result['passes']:
    print('✅ SNR TEST PASSED — ITrend reduces more noise than SMA')
else:
    print('❌ SNR TEST FAILED — SMA is smoother than ITrend')

# Plot residuals (noise) for both — visually confirms the SNR numbers
# Smaller residual = less noise = better indicator
sma = df_fx['Close'].rolling(window=PERIOD).mean()
noise_itrend = df_fx['Close'] - df_fx['ITrend']
noise_sma    = df_fx['Close'] - sma

fig, axes = plt.subplots(2, 1, figsize=(16, 6), sharex=True)
axes[0].plot(df_fx.index, noise_itrend, color='blue', linewidth=0.7, label='Noise (ITrend)')
axes[0].axhline(0, color='black', linewidth=0.5)
axes[0].set_title('Residual noise — ITrend')
axes[0].legend()

axes[1].plot(df_fx.index, noise_sma, color='green', linewidth=0.7, label=f'Noise (SMA{PERIOD})')
axes[1].axhline(0, color='black', linewidth=0.5)
axes[1].set_title(f'Residual noise — SMA{PERIOD}')
axes[1].legend()

plt.tight_layout()
plt.show()

## Section 6 — FX vs Crypto Comparison
Run this section only after the crypto CSV export exists.
Comment out if crypto data is not yet available.

In [ ]:
# [UNCOMMENT WHEN CRYPTO CSV IS READY]

# df_crypto = load_zorro_csv(CRYPTO_CSV)
#
# cross = compare_fx_crypto(
#     df_fx['Close'],    df_fx['ITrend'],
#     df_crypto['Close'], df_crypto['ITrend'],
#     PERIOD
# )
#
# print(f'FX    lag : {cross["fx_lag_bars"]} bars')
# print(f'Crypto lag: {cross["crypto_lag_bars"]} bars')
# print(f'FX    SNR improvement: {cross["fx_snr_improvement"]}x')
# print(f'Crypto SNR improvement: {cross["crypto_snr_improvement"]}x')
# print(f'Recommendation: {cross["recommendation"]}')

print('Section 6 placeholder — uncomment when crypto CSV is ready')

## Section 7 — Validation Verdict
Final PASS/FAIL. Run this last.

In [ ]:
# Full validation report — FX only for now
# When crypto data is ready, replace the placeholder series with real crypto data

# Placeholder crypto series (copy of FX) — replace when crypto CSV exists
# This allows the report function to run even before crypto data is available
# but crypto criteria will not be meaningful until real data is loaded
crypto_placeholder = df_fx['Close'].copy()
crypto_ind_placeholder = df_fx['ITrend'].copy()

report = generate_validation_report(
    indicator_name   = '[FILL IN]',   # e.g. 'InstantTrendline'
    price_fx         = df_fx['Close'],
    indicator_fx     = df_fx['ITrend'],
    price_crypto     = crypto_placeholder,
    indicator_crypto = crypto_ind_placeholder,
    period           = PERIOD
)

# Store report for later comparison across all indicators
# In Week 14 you will load all saved reports and build the master comparison table
import json
report_serializable = {
    k: v for k, v in report.items()
    if not isinstance(v, dict)  # exclude nested dicts for simple JSON
}
print('\nReport stored in variable: report')

## Section 8 — Notes & Parameter Recommendations
Fill in after all sections above are complete.

**Lag result:** [X bars — better/worse/same as SMA]

**SNR result:** [Xx improvement — passes/fails]

**FX recommended period:** [FILL IN]

**Crypto recommended period:** [FILL IN — adjust if lag was worse on crypto]

**NNFX role confirmed:** [Baseline / C1 / C2 / Exit / ATR — update from Section 1 hypothesis]

**Known limitations:** [Edge cases, market regimes where it struggles]

**Overall verdict:** [PASS / FAIL + one sentence reason]